In [ ]:
import re
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken
import torch.nn as nn

In [ ]:

class SimplyTokenizer:
    def __init__(self, vocab:dict):
        self.str2int = vocab
        self.int2str = {i:s for s,i in vocab.items()} #错了，s, i in vocab只是遍历键
    def encode(self, text):
        text = text.strip()
        text_list = re.split(r'([<>?/:,.!;\'"]|--|\n)', text)
        text_list = [word.strip() for word in text_list if word.strip()]#这里for循环可以写在前面，是因为if xxx.strip()只是过滤条件，而下面是在赋值
        encoded = [
            self.str2int[token] if token in self.str2int else self.str2int["<|unk|>"]
            for token in text_list#for循环放在最后，要放进列表的结果 for 临时变量 in 可迭代对象
        ] #又错了，self.str2int[token]在没找到时只会报keyerror，走不到else分支；如果self.str2int[token]返回id为0（关键），也会被认为false退出。所以不能用
        return encoded
    def decode(self, token_ids):
        raw = " ".join([self.int2str[idx] for idx in token_ids]) #知道这个地方要保留两边空格，但忘记这个join函数了
        decode_text = re.sub(r'\s+([,.;:?_\'"!()]|--)', r'\1', raw)#\s在正则表达式里表示特殊字符，xxx+表示前面的符号匹配多次，（）表示捕获组，r'\1'放在repl代表保留第一个捕获组。我提示的地方都写错了
        return decode_text
class GPTDataSet(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        super().__init__()
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text) #[T]
        for i in range(0, len(token_ids)-context_length, stride): #这个地方忘记用range了，在想token_ids[:,-context_size:]啥的，好像和计算logits那里混了
            input_id = token_ids[i : i + context_length]
            target_id = token_ids[i+1 : i + context_length + 1]
            self.input_ids.append(input_id) #tensor(input_id)相比于input_id更好，因为dataloader迭代堆栈时需要用tensor
            self.target_ids.append(target_id)
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
#取数据集
def create_data_loader(text,
                       tokenizer,
                       batch_size=4,
                       context_length=256,
                       stride=128,
                       shuffle=True,
                       drop_last=True,
                       num_workers=0): #用函数是因为dataloader class本身就是成熟的图纸了，不需要再自己加一些东西。只需要用函数流程进行封装
    dataset = GPTDataSet(text, tokenizer, context_length, stride)
    dataloder = DataLoader(dataset, 
                           batch_size, 
                           shuffle=shuffle,
                           drop_last=drop_last,
                           num_workers=num_workers)
    return dataloder

In [ ]:
#一步到位实现MHA
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, context_length,qkv_bias=False) -> None:
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divided by num_heads"
        self.num_heads = num_heads
        self.d_out = d_out
        self.W_q = nn.Linear(d_in, d_out, qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length,context_length,dtype=torch.bool),diagonal=1
            )#得把主对角线保留，所以要加diagonal
            )
        self.out_proj = nn.Linear(d_out, d_out)#在context_vec出来后，通过这个矩阵整理归类多头的信息
    def forward(self, x):
        b, t, i = x.shape
        d = self.d_out
        h = d // self.num_heads #又错了，/浮点数，view一定要init否则报错
        queries = self.W_q(x).view(b, t, self.num_heads, h).transpose(1, 2)#[B,N,T,H]
        keys = self.W_k(x).view(b, t, self.num_heads, h).transpose(1, 2)
        values = self.W_v(x).view(b, t, self.num_heads, h).transpose(1, 2)
        att_s = queries @ keys.transpose(2, 3) / keys.shape[-1]**0.5# torch.sqrt(keys.shape[-1])会报错，因为keys.shape[-1] 是 Python int，torch.sqrt() 需要 tensor
        mask = self.mask[:t, :t] #这里之前忘记写了，没细想直接略过了
        att_s.masked_fill_(mask, -torch.inf)
        att_w = torch.softmax(att_s, -1)
        att_w = self.dropout(att_w)
        context_vec = att_w @ values
        return self.out_proj(context_vec.transpose(1,2).contiguous().view(b, t, d))

总结一下今天的python知识点：
函数上：
join用于合并list的各个组件为str，前面是合并间隔元素
range用于迭代，前两个参数确定范围，后一个参数确定步长，参数需要是python int？大概确定需要迭代和int时使用
re在使用sub时，第一个参数传入匹配项，第二个参数传入替换项。/代表转义，re会识别相应转义项。()负责捕获，+没有\代表前面的元素匹配一个或者多个前面元素，[]代表一组内容
append方法是把小组件塞到大组件里
pytorch上：
torch.sqrt必须接受tensor。而tensor.shape[xx]返回python int
casual mask记得把主对角线保留，需要传入diagonal参数
torch.view接收的参数是int
语法上： 
取值 for 循环 for放在最后，过滤条件可以放在for循环后面
if 0、"",[], {}，none, set()等都会返回false
字典找不到key报错key error，会直接略过之后所有内容
要取字典的键值对需要用dict.items()语法
/返回float， //返回int

编程范式上：
class可以形象理解成一个万能图纸，class可以越写越细。就像图纸越来越精密
函数可以理解成一个流程。流程可以编在图纸里
其他的各种符号可以看作组装图纸和执行流程的实物

In [ ]:
#实现前馈神经网络
class GELU(nn.Module):
    def __init__(self) -> None:
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device)) * (x + 0.044715 * x**3)
        ))
class FeedFoward(nn.Module):
    def __init__(self, d_out, hidd_num=4) -> None:
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_out, hidd_num*d_out),
            GELU(),
            nn.Linear(hidd_num*d_out, d_out)
        )
    def forward(self, x):
        return self.ffn(x)

In [ ]:
#实现归一化层。错误总结：scale和bias是可学习的，意味着它要有梯度，同时要能被optimizer更新，前者要求require_grad为真，后者要求被注册为模型参数。同时，最好可学习参数表达能力尽可能地强，所以要求这两个值都是多维而不是常数
class LayerNorm(nn.Module):
    def __init__(self, d_in, eps=1e-5) -> None:
        super().__init__()
        self.scale=nn.Parameter(torch.ones(d_in))#self.scale = scale错误，只有被登记为parameter,optimizer才会更新，这种写法只是把数值挂在对象上，没有意义，达不到学习的效果。另外常数表达能力比torch.ones/zero更弱，因为全维度共享一个缩放。
        self.bias = nn.Parameter(torch.zeros(d_in))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)#又忘记把unbiased设置为False了，这里一定记住得用它，因为是总体样本方差
        x = (x-mean) / (var + self.eps)**0.5
        return self.scale*x + self.bias

In [ ]:
#实现transformerblock
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, drop_out, context_length, qkv_bias=False) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(emb_dim, emb_dim, drop_out, num_heads, context_length, qkv_bias)
        self.ffn = FeedFoward(emb_dim)
        self.norm1 = LayerNorm(emb_dim)
        self.norm2 = LayerNorm(emb_dim)
        self.dropout = nn.Dropout(drop_out)
    def forward(self, x):
        shortcut = x
        x = self.norm1(x) #错误，残差应该在归一化之前加shortcut，这表示神经网络只学增量信息。好处是梯度更容易传、深层网络稳定、每层只学修正量不必重建全部信息，鲁棒性高，某层没学好不至于完全破坏主干表示。缺点是每层输出保留大量旧信息，模型可能需要后续层慢慢修正；残差太强会导致学习效果不佳；没有norm、dropout以及初始化配合，残差叠加会导致输出尺度过大。
        #mha
        x = self.mha(x)
        x = self.dropout(x)
        x = x + shortcut
        #ffn
        shortcut = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x = x + shortcut
        return x
        

In [ ]:
#实现GPT
class GPTModel(nn.Module):
    def __init__(self, emb_dim, num_heads, drop_out, context_length, vocab_size, layer, qkv_bias=False) -> None:
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(context_length, emb_dim)
        self.out = nn.Linear(emb_dim, vocab_size, bias=False)
        '''self.transformer = nn.Sequential(
            TransformerBlock(emb_dim, emb_dim, num_heads, drop_out, context_length, qkv_bias)*layer
        )错的，用*[]来解包可迭代对象，里面放transformer模块，而不是直接乘，因为模块不能直接乘整数。要创建多层，必须真的创建多个对象'''
        self.transformer = nn.Sequential(
            *[TransformerBlock(emb_dim, num_heads, drop_out, context_length, qkv_bias) 
             for _ in range(layer)]
        )
        self.drop_emb = nn.Dropout(drop_out) #emb通常也需要drop,防止模型过度依赖某些token embedding或position embedding组合，让nn学到更分散、鲁棒的特征组合
        self.finorm = LayerNorm(emb_dim)
    def forward(self, token_ids):
        b, s = token_ids.shape
        emb1 = self.token_emb(token_ids)
        pos_ids = torch.arange(s, device=token_ids.device)#emb2 = self.pos_emb([i for i in range(s)])错，nn.Embedding需要tensor索引不能传入python list
        emb2 = self.pos_emb(pos_ids)
        emb = emb1 + emb2
        emb = self.drop_emb(emb)
        new_vec = self.finorm(self.transformer(emb))
        return self.out(new_vec)        